<a href="https://colab.research.google.com/github/Swag-Pseudopy/Escaping-Preference-Collapse/blob/main/nlhf_figures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
nlhf_figures.py  —  generates the 6 missing figures for the NLHF paper.

Outputs (./Images/):
  fig_initdist.pdf         Lemma 3  : log-scale initial energy distribution
  fig_bifurcation.pdf      Theorem 1: multi-seed bifurcation + C* prediction
  fig_gpt2_timeseries.pdf  GPT-2    : time-series three regimes
  fig_gpt2_bifdiagram.pdf  GPT-2    : C* vs tau/eta with (tau/eta)^{-1} fit
  fig_highdim.pdf          N=50     : three regimes in random tournament
  fig_mlp.pdf              MLP      : phase transition in output layer

Dependencies: numpy matplotlib torch  (transformers optional, set USE_GPT2=False to skip)
"""

import os, warnings
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.makedirs("Images", exist_ok=True)

USE_GPT2 = True   # set False if transformers not installed

plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.labelsize": 10,   "axes.titlesize": 10,
    "legend.fontsize": 8,   "lines.linewidth": 1.6,
    "figure.dpi": 220,
})
RED, GREEN, BLUE = "#d62728", "#2ca02c", "#1f77b4"
NASH_EQ = "#555555"

RNG = np.random.default_rng(42)

# ─────────────────────────────────────────────────────────────────────────────
# Shared helpers
# ─────────────────────────────────────────────────────────────────────────────

def condorcet_A(k: int) -> np.ndarray:
    A = np.zeros((k, k))
    for i in range(k):
        A[i, (i+1) % k] =  1.0
        A[(i+1) % k, i] = -1.0
    return A

def energy(p: np.ndarray) -> float:
    """V(p) = -sum log p_i  (excess = V - C_min)."""
    k = len(p)
    return float(-np.sum(np.log(np.clip(p, 1e-15, None))) - k * np.log(k))

def nash_md_step(p, A, eta, tau, p_ref, noise_std=0.0):
    """One regularised Nash-MD step."""
    Ap = A @ p
    if noise_std > 0:
        Ap = Ap + RNG.normal(0, noise_std, size=Ap.shape)
    log_p_new = (1 - eta*tau)*np.log(np.clip(p, 1e-15, None)) \
                + eta*tau*np.log(np.clip(p_ref, 1e-15, None)) \
                + eta * Ap
    log_p_new -= np.max(log_p_new)          # numerical stability
    p_new = np.exp(log_p_new)
    return p_new / p_new.sum()

def run_nash_md(p0, A, eta, tau, T=1500, noise_std=0.0):
    k = len(p0)
    p_ref = np.ones(k) / k
    p = p0.copy()
    C = [energy(p)]
    for _ in range(T - 1):
        p = nash_md_step(p, A, eta, tau, p_ref, noise_std)
        C.append(energy(p))
    return np.array(C)

def e_exp(p, A):
    Ap = A @ p
    return 1.5 * float(np.sum(p * Ap**2))

# ─────────────────────────────────────────────────────────────────────────────
# 1. fig_initdist  —  Lemma 3: log-scale initial energy distribution
# ─────────────────────────────────────────────────────────────────────────────

def fig_initdist():
    N = 50_000
    # uniform Dirichlet(1,1,1) on Delta^2
    raw = RNG.exponential(1.0, size=(N, 3))
    P   = raw / raw.sum(axis=1, keepdims=True)
    C   = -np.sum(np.log(np.clip(P, 1e-15, None)), axis=1) - 3*np.log(3)
    C   = C[C > 0]

    t_vals = np.linspace(0, C.max(), 300)
    theory = t_vals * np.exp(-t_vals / 3)
    theory = theory / theory.max() * (np.histogram(C, bins=60)[0].max())

    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    counts, edges, _ = ax.hist(C, bins=60, color=BLUE, alpha=0.7,
                               density=False, label="Empirical")
    ax.set_yscale("log")
    ax.plot(t_vals, theory, color=RED, lw=2,
            label=r"$\mathcal{O}(t\,e^{-t/3})$")
    ax.set_xlabel(r"Excess energy $\tilde{C}$")
    ax.set_ylabel("Count (log scale)")
    ax.set_title("Initial Cycle-Energy Distribution (Lemma 3)")
    ax.legend()
    fig.tight_layout()
    fig.savefig("Images/fig_initdist.pdf")
    plt.close(fig)
    print("✓  fig_initdist.pdf")

# ─────────────────────────────────────────────────────────────────────────────
# 2. fig_bifurcation  —  Theorem 1: mean ±1σ over 20 seeds + predicted C*
# ─────────────────────────────────────────────────────────────────────────────

def fig_bifurcation():
    A   = condorcet_A(3)
    eta = 0.05
    T   = 1500
    SEEDS = 20

    configs = [
        (0.002, "Expansive",  RED),
        (0.025, "Stable",     GREEN),
        (0.200, "Collapsing", BLUE),
    ]

    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    iters = np.arange(T)

    for tau, label, color in configs:
        runs = []
        for seed in range(SEEDS):
            rng = np.random.default_rng(seed)
            raw = rng.exponential(1.0, 3)
            p0  = raw / raw.sum()
            runs.append(run_nash_md(p0, A, eta, tau, T))
        runs = np.array(runs)          # (SEEDS, T)
        mu   = runs.mean(axis=0)
        sig  = runs.std(axis=0)
        ax.plot(iters, mu, color=color, label=rf"$\tau={tau}$  ({label})")
        ax.fill_between(iters, mu-sig, mu+sig, color=color, alpha=0.18)

    # Predicted C* for stable regime (tau=kappa*eta => kappa=0.5)
    p_star = np.ones(3)/3
    Cstar  = e_exp(p_star, A) / (0.025 / eta)   # = E_exp / kappa
    ax.axhline(Cstar, color=GREEN, ls="--", lw=1.2,
               label=rf"Predicted $\tilde{{C}}^*={Cstar:.3f}$")
    ax.axhline(0,     color=NASH_EQ, ls=":", lw=1.0, label="Nash eq.")

    ax.set_xlabel("Iteration")
    ax.set_ylabel(r"Excess energy $\tilde{C}_t$")
    ax.set_title(r"Stability Bifurcation — mean $\pm1\sigma$, 20 seeds (Theorem 1)")
    ax.set_ylim(-0.05, None)
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig("Images/fig_bifurcation.pdf")
    plt.close(fig)
    print("✓  fig_bifurcation.pdf")

# ─────────────────────────────────────────────────────────────────────────────
# 3 & 4.  GPT-2 figures  —  time-series and bifurcation diagram
#   Uses GPT-2 log-probs to initialise p(0) over k=5 response templates.
#   If USE_GPT2=False, falls back to a non-uniform Dirichlet prior that
#   produces identical dynamics (clearly labeled in the figure).
# ─────────────────────────────────────────────────────────────────────────────

TEMPLATES_5 = [
    "Explain neural networks concisely.",
    "Explain neural networks in detail.",
    "Explain neural networks with bullet points.",
    "Explain neural networks with an analogy.",
    "Explain neural networks technically.",
]

def get_gpt2_prior(k=5):
    """Returns softmax of GPT-2 log-probs over k response templates."""
    if not USE_GPT2:
        return None
    try:
        import torch
        from transformers import GPT2LMHeadModel, GPT2Tokenizer
        tok   = GPT2Tokenizer.from_pretrained("gpt2")
        model = GPT2LMHeadModel.from_pretrained("gpt2")
        model.eval()
        log_probs = []
        with torch.no_grad():
            for txt in TEMPLATES_5[:k]:
                ids = tok(txt, return_tensors="pt").input_ids
                out = model(ids, labels=ids)
                log_probs.append(-out.loss.item())   # mean token log-prob
        lp = np.array(log_probs, dtype=float)
        lp -= lp.max()
        p0 = np.exp(lp); p0 /= p0.sum()
        return p0
    except Exception as e:
        print(f"  GPT-2 unavailable ({e}), using non-uniform Dirichlet prior.")
        return None

def fig_gpt2_timeseries():
    k    = 5
    A    = condorcet_A(k)
    eta  = 0.05
    T    = 1500

    p0_gpt2 = get_gpt2_prior(k)
    if p0_gpt2 is not None:
        p0    = p0_gpt2
        label = "GPT-2 prior"
    else:
        raw   = RNG.dirichlet([0.5]*k)
        p0    = raw
        label = "Non-uniform Dirichlet prior"

    configs = [
        (0.005, "Expansive",  RED),
        (0.05,  "Stable",     GREEN),
        (0.50,  "Collapsing", BLUE),
    ]

    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    iters   = np.arange(T)
    for tau, lbl, color in configs:
        C = run_nash_md(p0, A, eta, tau, T)
        ax.plot(iters, C, color=color, label=rf"$\tau={tau}$ ({lbl})")
    ax.axhline(0, color=NASH_EQ, ls=":", lw=1.0, label="Nash eq.")
    ax.set_xlabel("Iteration")
    ax.set_ylabel(r"Excess energy $\tilde{C}_t$")
    ax.set_title(rf"$k=5$ Condorcet cycle — {label}" + "\n(time-series, three regimes)")
    ax.legend()
    fig.tight_layout()
    fig.savefig("Images/fig_gpt2_timeseries.pdf")
    plt.close(fig)
    print("✓  fig_gpt2_timeseries.pdf")

def fig_gpt2_bifdiagram():
    k    = 5
    A    = condorcet_A(k)
    eta  = 0.05
    T    = 1500
    LAST = 300    # average last 300 iters for steady-state C*

    p0_gpt2 = get_gpt2_prior(k)
    if p0_gpt2 is not None:
        p0    = p0_gpt2
        label = "GPT-2 prior"
    else:
        raw   = RNG.dirichlet([0.5]*k)
        p0    = raw
        label = "Non-uniform Dirichlet prior"

    tau_vals  = np.logspace(-2, 1, 40)   # tau/eta from 0.2 to 200
    cstar_emp = []
    for tau in tau_vals:
        C = run_nash_md(p0, A, eta, tau, T)
        cstar_emp.append(np.mean(C[-LAST:]))
    cstar_emp = np.array(cstar_emp)

    ratio     = tau_vals / eta
    # Theoretical curve: C* = E_exp(p*) / kappa = E_exp(p*) * eta / tau
    p_star    = np.ones(k) / k
    Eexp_star = (k/2) * np.sum(p_star * (A @ p_star)**2)
    cstar_theory = Eexp_star * eta / tau_vals

    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    ax.loglog(ratio, cstar_emp,    "o", ms=4, color=BLUE, label="Empirical $\\tilde{C}^*$")
    ax.loglog(ratio, cstar_theory, "-", color=RED,
              label=r"Theorem 1: $\tilde{C}^*\propto(\tau/\eta)^{-1}$")
    ax.set_xlabel(r"$\tau/\eta$")
    ax.set_ylabel(r"Steady-state $\tilde{C}^*$")
    ax.set_title(f"Bifurcation Diagram — {label}")
    ax.legend()
    fig.tight_layout()
    fig.savefig("Images/fig_gpt2_bifdiagram.pdf")
    plt.close(fig)
    print("✓  fig_gpt2_bifdiagram.pdf")

# ─────────────────────────────────────────────────────────────────────────────
# 5. fig_highdim  —  N=50 random skew-symmetric tournament
# ─────────────────────────────────────────────────────────────────────────────

def fig_highdim():
    N = 50
    raw = RNG.uniform(0, 1, (N, N))
    A   = raw - raw.T                 # skew-symmetric
    A  /= np.abs(A).max()             # normalise to [-1,1]

    eta = 0.01                        # smaller eta for larger ||A||
    T   = 2000

    raw_p0 = RNG.exponential(1.0, N)
    p0     = raw_p0 / raw_p0.sum()

    configs = [
        (0.0005, "Expansive",  RED),
        (0.005,  "Stable",     GREEN),
        (0.05,   "Collapsing", BLUE),
    ]

    fig, ax = plt.subplots(figsize=(5.0, 3.2))
    iters   = np.arange(T)
    for tau, lbl, color in configs:
        C = run_nash_md(p0, A, eta, tau, T)
        ax.plot(iters, C, color=color, label=rf"$\tau={tau}$ ({lbl})")
    ax.axhline(0, color=NASH_EQ, ls=":", lw=1.0, label="Nash eq.")
    ax.set_xlabel("Iteration")
    ax.set_ylabel(r"Excess energy $\tilde{C}_t$")
    ax.set_title(r"$N=50$ Random Tournament (Proposition 1)")
    ax.legend()
    fig.tight_layout()
    fig.savefig("Images/fig_highdim.pdf")
    plt.close(fig)
    print("✓  fig_highdim.pdf")

# ─────────────────────────────────────────────────────────────────────────────
# 6. fig_mlp  —  PyTorch MLP output-layer phase transition
# ─────────────────────────────────────────────────────────────────────────────

def fig_mlp():
    import torch
    import torch.nn as nn

    torch.manual_seed(0)

    class PolicyMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(10, 32), nn.ReLU(),
                nn.Linear(32, 3),
            )
        def forward(self, x):
            return torch.softmax(self.net(x), dim=-1)

    A_np  = condorcet_A(3)
    A_t   = torch.tensor(A_np, dtype=torch.float32)
    x_ctx = torch.randn(1, 10)          # fixed random context
    p_ref = torch.ones(3) / 3

    configs = [
        (0.002, "Expansive",  RED),
        (0.025, "Stable",     GREEN),
        (0.200, "Collapsing", BLUE),
    ]

    eta = 0.05
    T   = 500

    fig, ax = plt.subplots(figsize=(5.0, 3.2))
    iters = np.arange(T)

    for tau, lbl, color in configs:
        model = PolicyMLP()
        opt   = torch.optim.SGD(model.parameters(), lr=eta)
        C_log = []

        for _ in range(T):
            p = model(x_ctx).squeeze()          # (3,)

            # Nash-MD loss: KL towards regularised target
            Ap      = A_t @ p.detach()
            log_tgt = (1 - eta*tau)*torch.log(p.detach().clamp(1e-12)) \
                      + eta*tau*torch.log(p_ref) \
                      + eta * Ap
            log_tgt = log_tgt - torch.logsumexp(log_tgt, dim=0)
            loss    = nn.functional.kl_div(torch.log(p.clamp(1e-12)),
                                           log_tgt.exp(), reduction="sum")

            opt.zero_grad(); loss.backward(); opt.step()

            with torch.no_grad():
                p_np = model(x_ctx).squeeze().numpy()
                C_log.append(energy(p_np))

        ax.plot(iters, C_log, color=color, label=rf"$\tau={tau}$ ({lbl})")

    ax.axhline(0, color=NASH_EQ, ls=":", lw=1.0, label="Nash eq.")
    ax.set_xlabel("Gradient step")
    ax.set_ylabel(r"Excess energy $\tilde{C}_t$")
    ax.set_title("MLP Output Layer — Phase Transition")
    ax.legend()
    fig.tight_layout()
    fig.savefig("Images/fig_mlp.pdf")
    plt.close(fig)
    print("✓  fig_mlp.pdf")

# ─────────────────────────────────────────────────────────────────────────────
# Run all
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("Generating figures...")
    fig_initdist()
    fig_bifurcation()
    fig_gpt2_timeseries()
    fig_gpt2_bifdiagram()
    fig_highdim()
    fig_mlp()
    print("\nAll figures saved to ./Images/")

Generating figures...
✓  fig_initdist.pdf
✓  fig_bifurcation.pdf


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


✓  fig_gpt2_timeseries.pdf


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓  fig_gpt2_bifdiagram.pdf
✓  fig_highdim.pdf
✓  fig_mlp.pdf

All figures saved to ./Images/
